# Alpha Sweep Transfer: DW-Trunc1 GPU Campaign

Colab data-generation notebook for the DW-truncated alpha sweep with both no-feedback and perfect-correction variants. The campaign uses the canonical GPU dynamics entry point `classA_U1FGTN_gpu.run_markov_circuit(...)`, targets an A100 40 GB runtime, and writes CPU-compatible alpha-sweep outputs.


In [ ]:
from __future__ import annotations

from contextlib import contextmanager
import csv
import hashlib
import importlib.util
import inspect
import json
import math
import os
import sys
import tempfile
import threading
import time
from dataclasses import dataclass
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_no_feedback_alpha_sweep_transfer'
BUNDLE_ROOT_OVERRIDE = None  # e.g. '/content/drive/MyDrive/my_project/colab_no_feedback_alpha_sweep_transfer'

def is_bundle_root(path: Path) -> bool:
    return (
        (path / 'src' / 'classA_U1FGTN_gpu.py').exists()
        and (path / 'src' / 'choi_transfer_observables_gpu.py').exists()
    )

def find_bundle_root() -> Path:
    candidates: list[Path] = []
    if BUNDLE_ROOT_OVERRIDE is not None:
        candidates.append(Path(BUNDLE_ROOT_OVERRIDE).expanduser())
    for base in [Path.cwd(), *Path.cwd().parents]:
        candidates.extend([base, base / BUNDLE_NAME])
    if IN_COLAB:
        drive_root = Path('/content/drive/MyDrive')
        candidates.extend([
            drive_root / BUNDLE_NAME,
            drive_root / 'class_A_fermionic_adaptive_circuit' / BUNDLE_NAME,
        ])
        candidates.extend(path for path in drive_root.rglob(BUNDLE_NAME) if path.is_dir())
    checked: list[Path] = []
    for candidate in candidates:
        candidate = candidate.resolve()
        if candidate in checked:
            continue
        checked.append(candidate)
        if is_bundle_root(candidate):
            return candidate
    raise FileNotFoundError(
        f'Could not locate {BUNDLE_NAME} with src/classA_U1FGTN_gpu.py and '
        'src/choi_transfer_observables_gpu.py. Upload the complete bundle to Drive '
        'or set BUNDLE_ROOT_OVERRIDE.'
    )

BUNDLE_ROOT = find_bundle_root()
SRC_ROOT = BUNDLE_ROOT / 'src'
ENGINE_PATH = SRC_ROOT / 'classA_U1FGTN_gpu.py'
OBSERVER_PATH = SRC_ROOT / 'choi_transfer_observables_gpu.py'

def load_bundle_module(module_name: str, module_path: Path):
    if not module_path.is_file():
        raise FileNotFoundError(f'Missing required bundle module: {module_path}')
    spec = importlib.util.spec_from_file_location(module_name, module_path)
    if spec is None or spec.loader is None:
        raise ImportError(f'Could not create import specification for {module_path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module

import numpy as np
import pandas as pd
import torch
from IPython.display import Javascript, Markdown, display
from tqdm.auto import tqdm

HEARTBEAT_INTERVAL_S = 60.0


@contextmanager
def progress_heartbeat(progress_bar, config_label: str, *, interval_s: float = HEARTBEAT_INTERVAL_S):
    if interval_s <= 0:
        raise ValueError('heartbeat interval must be positive')
    stop_event = threading.Event()
    started = time.perf_counter()

    def emit() -> None:
        while not stop_event.wait(interval_s):
            elapsed_s = time.perf_counter() - started
            progress_bar.set_postfix_str(f'{config_label} elapsed={elapsed_s:.0f}s', refresh=True)
            progress_bar.write(f'[heartbeat] {config_label} still running; elapsed={elapsed_s:.0f}s')

    worker = threading.Thread(target=emit, name='colab-progress-heartbeat', daemon=True)
    worker.start()
    try:
        yield
    finally:
        stop_event.set()
        worker.join()

engine_module = load_bundle_module('classA_U1FGTN_gpu', ENGINE_PATH)
observer_module = load_bundle_module('choi_transfer_observables_gpu', OBSERVER_PATH)
classA_U1FGTN_gpu = engine_module.classA_U1FGTN_gpu
CANONICAL_ENTRY_POINT = observer_module.CANONICAL_ENTRY_POINT
CHOI_FORMULA_VERSION = observer_module.CHOI_FORMULA_VERSION
FINAL_SPECTRUM_EXTRACTION = observer_module.FINAL_SPECTRUM_EXTRACTION
GAP_DEFINITION = observer_module.GAP_DEFINITION
GAP_EXTRACTION = observer_module.GAP_EXTRACTION
HELPER_VERSION = observer_module.HELPER_VERSION
OBSERVABLE = observer_module.OBSERVABLE
PARTICLE_TRANSFER_DEFINITION = observer_module.PARTICLE_TRANSFER_DEFINITION
ParticleChoiGapObserver = observer_module.ParticleChoiGapObserver
exact_particle_spectrum_from_choi_blocks = observer_module.exact_particle_spectrum_from_choi_blocks
matrix_free_near_gap_single = observer_module.matrix_free_near_gap_single

if OBSERVABLE != 'complex_particle_choi_transfer_gap':
    raise RuntimeError(f'Unexpected observable helper: {OBSERVABLE}')
signature = inspect.signature(classA_U1FGTN_gpu.run_markov_circuit)
for name in ('track_choi', 'choi_observer', 'choi_observer_cycles', 'choi_singular_tol', 'choi_failure_mode', 'meas_slab_only'):
    if name not in signature.parameters:
        raise RuntimeError(f'Stale GPU engine missing {name}: {signature}')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this campaign.')
torch.set_grad_enabled(False)
print(f'[setup] bundle={BUNDLE_ROOT}')
print(f'[setup] engine={ENGINE_PATH}')
print(f'[setup] observer={OBSERVER_PATH}')
print(f'[setup] helper={HELPER_VERSION}, gpu={torch.cuda.get_device_name(0)}')
if torch.cuda.get_device_properties(0).total_memory / 1024**3 < 35.0:
    print('[warn] Production parameters assume an A100 40 GB runtime.')


In [ ]:
NX = 20
NY_VALUES = [20, 30, 40]
SAMPLES = 10
NSHELL = 1
ALPHA_VALUES = [1.0, 1.5, 1.8, 1.9, 2.0, 2.1, 2.2, 2.5, 3.0]
ALPHA_2 = 30.0
INIT_MODE = 'default'
DTYPE, DEVICE, BACKEND = 'complex128', 'cuda:0', 'local'
SEQUENCE = 'raster_y'
TRIAL_ORBITALS = 'X'
N_A = 0.5
P_GAIN = 0.0
P_LOSS = 0.0
CORRECTION_MODES = [
    {'correction_label': 'none', 'perfect_correction': False},
    {'correction_label': 'perfect', 'perfect_correction': True},
]
DW_LABEL = 'dwtrunc1'
DW_TRUNCATION = True
MEAS_SLAB_ONLY = True
BATCH_SIZE = None  # A100 40 GB auto-batching. Set to 1 for the smoke override.
OVERWRITE = False
SKIP_COMPLETED = True

ENDPOINT_TOL = 1e-12
CHOI_SPECTRAL_TOL = 1e-10
CHOI_SINGULAR_TOL = 1e-10
CHOI_FAILURE_MODE = 'censor'
CHOI_ENTRY_ABS_TOL = 1e-10
CHOI_HERMITICITY_TOL = 1e-9
CHOI_INVOLUTION_TOL = 1e-8
N_EIGENSTATES = 3
NEAR_GAP_BLOCK_SIZE = 8
NEAR_GAP_MAX_SUBSPACE = 32
NEAR_GAP_MAX_ITER = 80
NEAR_GAP_RESIDUAL_TOL = 1e-10

CAMPAIGN_NAME = 'alpha_sweep_transfer_dwtrunc1_no_feedback_and_perfect_correction'
CAMPAIGN_ID = 'N20_Ny20-30-40_nsh1_dwtrunc1_init-default_S10_cycles2Ny_alpha-sweep_corr-none-perfect'
GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
OUTPUT_ROOT = GPU_DATA_ROOT / CAMPAIGN_NAME
CAMPAIGN_ROOT = OUTPUT_ROOT / 'campaigns' / CAMPAIGN_ID
RUNS_ROOT = CAMPAIGN_ROOT / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

CONFIGS = [
    {
        'Nx': NX,
        'Ny': int(ny),
        'cycles': 2 * int(ny),
        'samples': SAMPLES,
        'nshell': NSHELL,
        'alpha_1': float(alpha_1),
        'alpha_2': ALPHA_2,
        'init_mode': INIT_MODE,
        'DW': True,
        'dw_label': DW_LABEL,
        'dw_truncation': DW_TRUNCATION,
        'meas_slab_only': MEAS_SLAB_ONLY,
        'n_a': N_A,
        'p_gain': P_GAIN,
        'p_loss': P_LOSS,
        'perfect_correction': bool(correction['perfect_correction']),
        'correction_label': str(correction['correction_label']),
    }
    for correction in CORRECTION_MODES
    for ny in NY_VALUES
    for alpha_1 in ALPHA_VALUES
]

config_df = pd.DataFrame(CONFIGS)
display(Markdown('### Campaign config'))
display(config_df[['Nx', 'Ny', 'cycles', 'samples', 'alpha_1', 'alpha_2', 'dw_label', 'init_mode', 'p_gain', 'p_loss', 'perfect_correction', 'correction_label']])
print(json.dumps({
    'campaign_name': CAMPAIGN_NAME,
    'campaign_id': CAMPAIGN_ID,
    'config_count': len(CONFIGS),
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'output_root': str(OUTPUT_ROOT),
}, indent=2))


## Mandatory CUDA Smoke Test

These checks exercise the Choi gap helper and a tiny `dwtrunc1` GPU runs for both correction modes before production.


In [ ]:
def synthetic_solver_checks() -> None:
    torch.manual_seed(111)
    n = 18
    q, _ = torch.linalg.qr(torch.randn((n, n), dtype=torch.complex128, device=DEVICE))
    vals = torch.tensor([-1.0, -0.7, -0.03, -0.002, 0.001, 0.04, 0.3, 1.0] + [0.5] * 10,
                        dtype=torch.float64, device=DEVICE)
    a = q @ torch.diag(vals.to(torch.complex128)) @ q.mH
    exact = exact_particle_spectrum_from_choi_blocks(
        a.unsqueeze(0), cycle=7, endpoint_tol=ENDPOINT_TOL,
        choi_spectral_tol=CHOI_SPECTRAL_TOL, n_eigenstates=3,
    )
    estimate = matrix_free_near_gap_single(
        a, cycle=7, endpoint_tol=ENDPOINT_TOL, choi_spectral_tol=CHOI_SPECTRAL_TOL,
        n_eigenstates=3, block_size=8, max_subspace=18, max_iter=80,
        residual_tol=1e-10, seed=5, exact_fallback=True,
    )
    assert torch.allclose(estimate['near_gap_exponents'], exact['near_gap_exponents'][0], atol=1e-9, rtol=1e-8)
    forced = matrix_free_near_gap_single(
        a, cycle=7, n_eigenstates=3, block_size=8, max_subspace=18,
        max_iter=0, residual_tol=1e-14, seed=5, exact_fallback=True,
    )
    assert bool(forced['used_exact_fallback'])
    assert torch.allclose(forced['gap'], exact['gap'][0])
    print('[smoke] synthetic matrix-free/exact and forced-fallback checks passed')

def run_dwtrunc1_smoke() -> None:
    samples = 2
    cycles = 2
    for correction in CORRECTION_MODES:
        label = str(correction['correction_label'])
        perfect = bool(correction['perfect_correction'])
        torch.manual_seed(1234)
        torch.cuda.manual_seed_all(1234)
        model = classA_U1FGTN_gpu(
            4, 8, DW=True, nshell=1, alpha_1=1.0, alpha_2=30.0,
            dw_truncation=True, device=DEVICE, dtype=DTYPE, backend=BACKEND,
        )
        active = model.active_top_layer_indices(meas_slab_only=True)
        observer = ParticleChoiGapObserver(
            samples_expected=samples, cycles=[cycles], final_cycle=cycles, nlayer=int(active.numel()),
            n_eigenstates=3, near_gap_block_size=8, near_gap_max_subspace=24,
            near_gap_max_iter=20, near_gap_residual_tol=1e-9, endpoint_tol=ENDPOINT_TOL,
            choi_spectral_tol=CHOI_SPECTRAL_TOL, validate_choi=True, censor_unstable=True,
            choi_entry_abs_tol=CHOI_ENTRY_ABS_TOL, choi_hermiticity_tol=CHOI_HERMITICITY_TOL,
            choi_involution_tol=CHOI_INVOLUTION_TOL,
            active_top_layer_indices=active.detach().cpu().numpy(), full_nlayer=model.Nlayer,
        )
        retained_sigma_ll = {}
        def smoke_observer(**payload):
            retained_sigma_ll[int(payload['cycle'])] = payload['sigma_ll'].detach().clone()
            return observer(**payload)
        result = model.run_markov_circuit(
            G_history=False, progress=False, cycles=cycles, samples=samples,
            postselect=False, postselect_probability=0.0, perfect_correction=perfect,
            save=False, return_data=False, n_a=N_A, p_gain=P_GAIN, p_loss=P_LOSS,
            sequence=SEQUENCE, meas_slab_only=True, batch_size=samples,
            track_choi=True, choi_observer=smoke_observer, choi_observer_cycles=[cycles],
            choi_singular_tol=CHOI_SINGULAR_TOL, choi_failure_mode=CHOI_FAILURE_MODE,
        )
        observer.assert_complete()
        assert bool(result['meas_slab_only_effective'])
        exact = exact_particle_spectrum_from_choi_blocks(
            retained_sigma_ll[cycles], cycle=cycles, endpoint_tol=ENDPOINT_TOL,
            choi_spectral_tol=CHOI_SPECTRAL_TOL, n_eigenstates=3,
        )
        assert np.max(np.abs(observer.gap[:, 0] - exact['gap'].detach().cpu().numpy())) < 1e-8
        assert np.nanmax(observer.hermiticity_residual) < 1e-9
        assert np.nanmax(observer.involution_residual) < 1e-8
        assert min(row['min_abs_d'] for row in result['choi_diagnostics']) > CHOI_SINGULAR_TOL
        print(f'[smoke] {label} DW-trunc1 Choi gap observer passed')
        del model, observer
        torch.cuda.empty_cache()

synthetic_solver_checks()
run_dwtrunc1_smoke()
torch.cuda.empty_cache()


## Production Campaign

The helpers below write CPU-compatible per-alpha sample files and campaign-level manifests.


In [ ]:
@dataclass(frozen=True)
class RunConfig:
    nx: int
    ny: int
    cycles: int
    samples: int
    nshell: int
    alpha_1: float
    alpha_2: float
    init_mode: str
    perfect_correction: bool
    correction_label: str

def utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()

def jsonable(value: Any) -> Any:
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, dict):
        return {str(key): jsonable(val) for key, val in value.items()}
    if isinstance(value, (list, tuple)):
        return [jsonable(item) for item in value]
    return value

def write_json_atomic(path: Path | str, payload: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f'.{path.name}.', suffix='.tmp', dir=path.parent)
    try:
        with os.fdopen(fd, 'w', encoding='utf-8') as fh:
            json.dump(jsonable(payload), fh, indent=2, sort_keys=True)
            fh.write('\n')
        os.replace(temp_name, path)
    finally:
        if os.path.exists(temp_name):
            os.unlink(temp_name)

def save_npz_atomic(path: Path | str, **arrays: Any) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f'.{path.stem}.', suffix='.npz', dir=path.parent)
    os.close(fd)
    try:
        np.savez_compressed(temp_name, **arrays)
        os.replace(temp_name, path)
    finally:
        if os.path.exists(temp_name):
            os.unlink(temp_name)

def write_csv_atomic(path: Path | str, rows: list[dict[str, Any]], fieldnames: list[str]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    fd, temp_name = tempfile.mkstemp(prefix=f'.{path.name}.', suffix='.tmp', dir=path.parent)
    try:
        with os.fdopen(fd, 'w', encoding='utf-8', newline='') as fh:
            writer = csv.DictWriter(fh, fieldnames=fieldnames)
            writer.writeheader()
            writer.writerows(rows)
        os.replace(temp_name, path)
    finally:
        if os.path.exists(temp_name):
            os.unlink(temp_name)

def alpha_tag(value: float) -> str:
    return f'{float(value):.6g}'.replace('-', 'm').replace('.', 'p')

def stable_seed(*parts: Any) -> int:
    digest = hashlib.sha256('|'.join(str(part) for part in parts).encode('utf-8')).digest()
    return int.from_bytes(digest[:4], 'little', signed=False)

def run_config_from_dict(cfg: dict[str, Any]) -> RunConfig:
    return RunConfig(
        nx=int(cfg['Nx']), ny=int(cfg['Ny']), cycles=int(cfg['cycles']), samples=int(cfg['samples']),
        nshell=int(cfg['nshell']), alpha_1=float(cfg['alpha_1']), alpha_2=float(cfg['alpha_2']),
        init_mode=str(cfg['init_mode']), perfect_correction=bool(cfg['perfect_correction']),
        correction_label=str(cfg['correction_label']),
    )

def run_dir_for(cfg: RunConfig) -> Path:
    return RUNS_ROOT / f'N{cfg.nx}x{cfg.ny}_{DW_LABEL}_corr_{cfg.correction_label}_alpha_{alpha_tag(cfg.alpha_1)}'

def samples_dir_for(run_dir: Path) -> Path:
    return run_dir / 'samples'

def sample_path(run_dir: Path, sample_index: int) -> Path:
    return samples_dir_for(run_dir) / f'sample_{int(sample_index):04d}.npz'

def expected_paths(run_dir: Path) -> dict[str, Path]:
    return {
        'summary_npz': run_dir / 'transfer_spectrum_summary.npz',
        'summary_csv': run_dir / 'transfer_gap_summary.csv',
        'run_summary': run_dir / 'run_summary.json',
        'latest': run_dir / 'latest_run.json',
    }

def config_seed(cfg: RunConfig) -> int:
    return stable_seed('no_feedback_alpha_sweep_transfer', 'gpu_config', cfg.init_mode, DW_LABEL,
                       cfg.correction_label, cfg.alpha_1, cfg.alpha_2, cfg.nx, cfg.ny, cfg.cycles, cfg.nshell, cfg.samples)

def sample_seed(cfg: RunConfig, sample_index: int) -> int:
    return stable_seed('no_feedback_alpha_sweep_transfer', cfg.init_mode, DW_LABEL, cfg.alpha_1,
                       cfg.correction_label, cfg.alpha_2, cfg.nx, cfg.ny, cfg.cycles, cfg.nshell, sample_index)

def transfer_a_from_scaled_exponents(scaled_exponents: np.ndarray) -> np.ndarray:
    scaled_exponents = np.asarray(scaled_exponents, dtype=np.float64)
    a_values = np.empty_like(scaled_exponents)
    a_values[np.isneginf(scaled_exponents)] = 1.0
    a_values[np.isposinf(scaled_exponents)] = -1.0
    finite = np.isfinite(scaled_exponents)
    a_values[finite] = -np.tanh(scaled_exponents[finite])
    a_values[np.isnan(scaled_exponents)] = np.nan
    return a_values

def completed_row(cfg: RunConfig, run_dir: Path) -> dict[str, Any] | None:
    latest_path = expected_paths(run_dir)['latest']
    if not SKIP_COMPLETED or not latest_path.exists():
        return None
    payload = json.loads(latest_path.read_text(encoding='utf-8'))
    row = payload.get('run_index_row')
    if (
        payload.get('status') == 'completed'
        and isinstance(row, dict)
        and row.get('Nx') == cfg.nx
        and row.get('Ny') == cfg.ny
        and float(row.get('alpha_1', 'nan')) == float(cfg.alpha_1)
        and row.get('dw_label') == DW_LABEL
        and row.get('sample_count') == cfg.samples
        and bool(row.get('perfect_correction')) == cfg.perfect_correction
        and row.get('correction_label') == cfg.correction_label
        and payload.get('helper_version') == HELPER_VERSION
    ):
        print(f'[skip] completed {cfg.correction_label} Ny={cfg.ny} alpha={cfg.alpha_1:g}')
        return row
    return None

def write_sample_outputs(cfg: RunConfig, run_dir: Path, observer: Any, metadata_base: dict[str, Any]) -> None:
    samples_dir_for(run_dir).mkdir(parents=True, exist_ok=True)
    final_idx = observer._cycle_to_index[cfg.cycles]
    final_exponents = np.asarray(observer.final_spectrum, dtype=np.float64)
    final_gap = np.asarray(observer.gap[:, final_idx], dtype=np.float64)
    final_stable = np.asarray(observer.stable_at_cycle[:, final_idx], dtype=np.bool_)
    min_abs_d = np.asarray(observer.min_abs_d[:, final_idx], dtype=np.float64)
    basis_size = int(final_exponents.shape[1])
    full_nlayer = -1 if observer.full_nlayer is None else int(observer.full_nlayer)
    with np.errstate(over='ignore', under='ignore', invalid='ignore'):
        singular_values = np.exp(float(cfg.cycles) * final_exponents)
    a_values = transfer_a_from_scaled_exponents(final_exponents * float(cfg.cycles))
    mode_a_values = np.asarray(observer.final_spectrum_a_eigenvalues, dtype=np.float64)
    mode_vectors = np.asarray(observer.final_spectrum_eigenstates, dtype=np.complex128)
    active_top_layer_indices = np.asarray(observer.active_top_layer_indices, dtype=np.int64) if observer.active_top_layer_indices is not None else np.arange(basis_size, dtype=np.int64)
    for sample_index in range(cfg.samples):
        seed = sample_seed(cfg, sample_index)
        failure_records = [record for record in observer.failure_details if int(record.get('sample_index', -1)) == sample_index]
        finite_mode_mask = np.isfinite(final_exponents[sample_index])
        finite_mode_indices = np.flatnonzero(finite_mode_mask).astype(np.int64)
        finite_mode_vectors = mode_vectors[sample_index][:, finite_mode_mask]
        sample_metadata = {**metadata_base, 'sample_index': int(sample_index), 'sample_seed': int(seed),
                           'choi_failure_records': failure_records,
                           'lyapunov_modes_saved': 'all finite final-cycle transfer modes',
                           'lyapunov_mode_axis_convention': 'finite_lyapunov_modes[:, k] is mode k in the reduced Choi slab basis'}
        save_npz_atomic(
            sample_path(run_dir, sample_index),
            alpha_1=np.asarray(cfg.alpha_1, dtype=np.float64),
            alpha_2=np.asarray(cfg.alpha_2, dtype=np.float64),
            dw_truncation=np.asarray(DW_TRUNCATION, dtype=bool),
            perfect_correction=np.asarray(cfg.perfect_correction, dtype=bool),
            correction_label=np.asarray(cfg.correction_label),
            meas_slab_only=np.asarray(MEAS_SLAB_ONLY, dtype=bool),
            sample_seed=np.asarray(seed, dtype=np.uint32),
            sample_index=np.asarray(sample_index, dtype=np.int64),
            cycle_count=np.asarray(cfg.cycles, dtype=np.int64),
            Nx=np.asarray(cfg.nx, dtype=np.int64),
            Ny=np.asarray(cfg.ny, dtype=np.int64),
            transfer_singular_values=singular_values[sample_index],
            transfer_exponents=final_exponents[sample_index],
            transfer_a_eigenvalues=a_values[sample_index],
            finite_lyapunov_mode_indices=finite_mode_indices,
            finite_lyapunov_exponents=final_exponents[sample_index, finite_mode_mask],
            finite_lyapunov_a_eigenvalues=mode_a_values[sample_index, finite_mode_mask],
            finite_lyapunov_modes=finite_mode_vectors,
            finite_lyapunov_mode_count=np.asarray(finite_mode_indices.size, dtype=np.int64),
            lyapunov_mode_basis_indices=active_top_layer_indices,
            transfer_gap=np.asarray(final_gap[sample_index], dtype=np.float64),
            finite_eigenstate_count=np.asarray(np.count_nonzero(np.isfinite(final_exponents[sample_index])), dtype=np.int64),
            particle_zero_count=np.asarray(observer.particle_zero_count[sample_index, final_idx], dtype=np.int64),
            particle_pole_count=np.asarray(observer.particle_pole_count[sample_index, final_idx], dtype=np.int64),
            a_eigenvalue_min=np.asarray(observer.a_eigenvalue_min_final[sample_index], dtype=np.float64),
            a_eigenvalue_max=np.asarray(observer.a_eigenvalue_max_final[sample_index], dtype=np.float64),
            choi_min_abs_d=np.asarray(min_abs_d[sample_index], dtype=np.float64),
            choi_active_final=np.asarray(final_stable[sample_index], dtype=bool),
            choi_basis_size=np.asarray(basis_size, dtype=np.int64),
            choi_full_nlayer=np.asarray(full_nlayer, dtype=np.int64),
            choi_failure_records_json=np.asarray(json.dumps(jsonable(failure_records), sort_keys=True)),
            metadata_json=np.asarray(json.dumps(jsonable(sample_metadata), sort_keys=True)),
        )

def aggregate_alpha(run_dir: Path, samples: int) -> dict[str, Any]:
    sample_paths = [sample_path(run_dir, idx) for idx in range(int(samples))]
    missing = [str(path) for path in sample_paths if not path.exists()]
    if missing:
        raise FileNotFoundError(f'Cannot aggregate; missing sample files: {missing[:5]}')
    sample_indices, seeds, singular_values, exponents, gaps = [], [], [], [], []
    finite_counts, min_abs_d, active_final, basis_size = [], [], [], []
    perfect_correction = False
    correction_label = ''
    alpha_1 = alpha_2 = float('nan')
    nx = ny = cycles = -1
    for path in sample_paths:
        with np.load(path, allow_pickle=False) as payload:
            alpha_1 = float(payload['alpha_1']); alpha_2 = float(payload['alpha_2'])
            perfect_correction = bool(payload['perfect_correction'])
            correction_label = str(payload['correction_label'])
            nx = int(payload['Nx']); ny = int(payload['Ny']); cycles = int(payload['cycle_count'])
            sample_indices.append(int(payload['sample_index'])); seeds.append(int(payload['sample_seed']))
            singular_values.append(np.asarray(payload['transfer_singular_values'], dtype=np.float64))
            exponents.append(np.asarray(payload['transfer_exponents'], dtype=np.float64))
            gaps.append(float(payload['transfer_gap']))
            finite_counts.append(int(payload['finite_eigenstate_count']))
            min_abs_d.append(float(payload['choi_min_abs_d']))
            active_final.append(bool(payload['choi_active_final']))
            basis_size.append(int(payload['choi_basis_size']))
    singular_stack = np.stack(singular_values, axis=0)
    exponent_stack = np.stack(exponents, axis=0)
    gap_arr = np.asarray(gaps, dtype=np.float64)
    gap_std = float(np.nanstd(gap_arr, ddof=1)) if gap_arr.size > 1 else 0.0
    summary = {
        'Nx': nx, 'Ny': ny, 'cycles': cycles, 'alpha_1': alpha_1, 'alpha_2': alpha_2,
        'perfect_correction': bool(perfect_correction), 'correction_label': correction_label,
        'sample_count': int(gap_arr.size),
        'transfer_gap_mean': float(np.nanmean(gap_arr)),
        'transfer_gap_std': gap_std,
        'transfer_gap_sem': float(gap_std / math.sqrt(gap_arr.size)) if gap_arr.size > 1 else 0.0,
        'transfer_gap_min': float(np.nanmin(gap_arr)),
        'transfer_gap_max': float(np.nanmax(gap_arr)),
        'finite_eigenstate_count_mean': float(np.mean(finite_counts)),
        'choi_active_fraction': float(np.mean(active_final)),
        'choi_basis_size': int(basis_size[0]) if basis_size else 0,
        'choi_min_abs_d_min': float(np.nanmin(np.asarray(min_abs_d, dtype=np.float64))),
    }
    save_npz_atomic(run_dir / 'transfer_spectrum_summary.npz',
        sample_index=np.asarray(sample_indices, dtype=np.int64), sample_seed=np.asarray(seeds, dtype=np.uint32),
        Nx=np.asarray(nx, dtype=np.int64), Ny=np.asarray(ny, dtype=np.int64), cycles=np.asarray(cycles, dtype=np.int64),
        alpha_1=np.asarray(alpha_1, dtype=np.float64), alpha_2=np.asarray(alpha_2, dtype=np.float64),
        perfect_correction=np.asarray(perfect_correction, dtype=bool), correction_label=np.asarray(correction_label),
        transfer_singular_values=singular_stack, transfer_exponents=exponent_stack, transfer_gap=gap_arr,
        finite_eigenstate_count=np.asarray(finite_counts, dtype=np.int64),
        choi_min_abs_d=np.asarray(min_abs_d, dtype=np.float64), choi_active_final=np.asarray(active_final, dtype=bool),
        choi_basis_size=np.asarray(basis_size, dtype=np.int64),
        transfer_singular_values_mean=np.nanmean(singular_stack, axis=0),
        transfer_exponents_mean=np.nanmean(exponent_stack, axis=0))
    write_csv_atomic(run_dir / 'transfer_gap_summary.csv', [summary], [
        'Nx', 'Ny', 'cycles', 'alpha_1', 'alpha_2', 'perfect_correction', 'correction_label', 'sample_count', 'transfer_gap_mean',
        'transfer_gap_std', 'transfer_gap_sem', 'transfer_gap_min', 'transfer_gap_max',
        'finite_eigenstate_count_mean', 'choi_active_fraction', 'choi_basis_size', 'choi_min_abs_d_min'])
    return summary

def write_run_index(path: Path, rows: list[dict[str, Any]]) -> None:
    write_csv_atomic(path, rows, [
        'config_id', 'Nx', 'Ny', 'cycles', 'alpha_1', 'alpha_2', 'dw_label', 'dw_truncation',
        'perfect_correction', 'correction_label', 'sample_count', 'stable_samples_final', 'transfer_gap_mean', 'transfer_gap_sem', 'elapsed_s',
        'batch_size', 'batch_size_mode', 'canonical_dynamics_entry_point', 'run_dir'])

def update_gpu_data_index(campaign_manifest: Path) -> None:
    index_path = GPU_DATA_ROOT / 'index.json'
    payload = json.loads(index_path.read_text(encoding='utf-8')) if index_path.exists() else {}
    payload['bundle_root'] = str(BUNDLE_ROOT)
    payload['gpu_data_root'] = str(GPU_DATA_ROOT)
    campaigns = payload.setdefault('campaigns', [])
    campaigns[:] = [item for item in campaigns if item.get('name') != CAMPAIGN_NAME]
    campaigns.append({
        'name': CAMPAIGN_NAME,
        'kind': 'gpu_alpha_sweep_transfer_dwtrunc1_no_feedback_and_perfect_correction',
        'manifest': str(campaign_manifest.relative_to(GPU_DATA_ROOT)),
        'latest': str((OUTPUT_ROOT / 'latest_campaign.json').relative_to(GPU_DATA_ROOT)),
    })
    write_json_atomic(index_path, payload)

def run_one_config(raw_cfg: dict[str, Any]) -> dict[str, Any]:
    cfg = run_config_from_dict(raw_cfg)
    run_dir = run_dir_for(cfg)
    old_row = completed_row(cfg, run_dir)
    if old_row is not None:
        return old_row
    run_dir.mkdir(parents=True, exist_ok=True)
    for path in expected_paths(run_dir).values():
        if path.exists() and not OVERWRITE:
            raise FileExistsError(f'Output already exists for Ny={cfg.ny}, alpha={cfg.alpha_1:g}: {path}')

    def execute(candidate_batch_size: int | None):
        torch.manual_seed(config_seed(cfg))
        torch.cuda.manual_seed_all(config_seed(cfg))
        model = classA_U1FGTN_gpu(
            Nx=cfg.nx, Ny=cfg.ny, DW=True, nshell=cfg.nshell, filling_frac=0.5,
            alpha_1=cfg.alpha_1, alpha_2=cfg.alpha_2, trial_orbitals=TRIAL_ORBITALS,
            dw_truncation=DW_TRUNCATION, triv_region_local_mode=False,
            device=DEVICE, dtype=DTYPE, backend=BACKEND)
        active = model.active_top_layer_indices(meas_slab_only=MEAS_SLAB_ONLY)
        observer = ParticleChoiGapObserver(
            samples_expected=cfg.samples, cycles=[cfg.cycles], final_cycle=cfg.cycles,
            nlayer=int(active.numel()), n_eigenstates=N_EIGENSTATES,
            near_gap_block_size=NEAR_GAP_BLOCK_SIZE, near_gap_max_subspace=NEAR_GAP_MAX_SUBSPACE,
            near_gap_max_iter=NEAR_GAP_MAX_ITER, near_gap_residual_tol=NEAR_GAP_RESIDUAL_TOL,
            endpoint_tol=ENDPOINT_TOL, choi_spectral_tol=CHOI_SPECTRAL_TOL, exact_fallback=True,
            validate_choi=False, censor_unstable=True, choi_entry_abs_tol=CHOI_ENTRY_ABS_TOL,
            choi_hermiticity_tol=CHOI_HERMITICITY_TOL, choi_involution_tol=CHOI_INVOLUTION_TOL,
            active_top_layer_indices=active.detach().cpu().numpy(), full_nlayer=model.Nlayer)
        started = time.perf_counter()
        result = model.run_markov_circuit(
            G_history=False, progress=False, cycles=cfg.cycles, samples=cfg.samples,
            postselect=False, postselect_probability=0.0, perfect_correction=cfg.perfect_correction,
            init_mode=cfg.init_mode, save=False, save_init=False, n_a=N_A, p_gain=P_GAIN, p_loss=P_LOSS,
            sequence=SEQUENCE, meas_slab_only=MEAS_SLAB_ONLY, batch_size=candidate_batch_size,
            return_data=False, track_choi=True, choi_observer=observer, choi_observer_cycles=[cfg.cycles],
            choi_singular_tol=CHOI_SINGULAR_TOL, choi_failure_mode=CHOI_FAILURE_MODE)
        torch.cuda.synchronize()
        elapsed_s = float(time.perf_counter() - started)
        observer.assert_complete()
        return model, observer, elapsed_s, result

    try:
        model, observer, elapsed_s, result = execute(BATCH_SIZE)
        retried_after_oom = False
    except RuntimeError as exc:
        if BATCH_SIZE is not None or 'out of memory' not in str(exc).lower():
            raise
        print(f'[warn] CUDA OOM for {cfg.correction_label} Ny={cfg.ny} alpha={cfg.alpha_1:g}; retrying with batch_size=1')
        torch.cuda.empty_cache()
        model, observer, elapsed_s, result = execute(1)
        retried_after_oom = True

    if not bool(result['meas_slab_only_effective']):
        raise RuntimeError('dwtrunc1 GPU run did not use the reduced topological slab basis.')
    metadata_base = {
        'experiment': CAMPAIGN_NAME, 'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'init_mode': cfg.init_mode, 'canonical_init_mode': cfg.init_mode,
        'sweep_config': {'dw_label': DW_LABEL, 'DW': True, 'dw_truncation': DW_TRUNCATION,
                         'meas_slab_only': MEAS_SLAB_ONLY, 'alpha_1': cfg.alpha_1, 'alpha_2': cfg.alpha_2,
                         'perfect_correction': cfg.perfect_correction, 'correction_label': cfg.correction_label},
        'Nx': cfg.nx, 'Ny': cfg.ny, 'cycles': cfg.cycles, 'nshell': cfg.nshell,
        'n_a': N_A, 'p_gain': P_GAIN, 'p_loss': P_LOSS, 'p_gain_effective': P_GAIN,
        'p_loss_effective': P_LOSS, 'perfect_correction': cfg.perfect_correction,
        'correction_label': cfg.correction_label, 'trial_orbitals': TRIAL_ORBITALS,
        'sequence': SEQUENCE, 'dtype': DTYPE, 'backend': BACKEND, 'started_at': utc_now(),
        'config_seed': int(config_seed(cfg)),
        'sample_seed_note': 'GPU engine uses a deterministic per-config Torch seed; sample_seed is a stable sample label.',
        'save_final_g': False, 'track_global_entropy': False, 'choi_basis': 'reduced_topological_slab',
        'helper_version': HELPER_VERSION, 'choi_formula_version': CHOI_FORMULA_VERSION,
        'gap_definition': GAP_DEFINITION, 'gap_extraction': GAP_EXTRACTION,
        'final_spectrum_extraction': FINAL_SPECTRUM_EXTRACTION,
    }
    write_sample_outputs(cfg, run_dir, observer, metadata_base)
    summary = aggregate_alpha(run_dir, cfg.samples)
    summary.update({'init_mode': cfg.init_mode, 'dw_label': DW_LABEL, 'dw_truncation': DW_TRUNCATION,
                    'meas_slab_only': MEAS_SLAB_ONLY, 'perfect_correction': cfg.perfect_correction,
                    'correction_label': cfg.correction_label, 'run_dir': str(run_dir)})
    run_summary = {
        **raw_cfg, 'status': 'completed', 'run_dir': str(run_dir),
        'sample_outputs': [str(sample_path(run_dir, idx)) for idx in range(cfg.samples)],
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT, 'helper_version': HELPER_VERSION,
        'choi_formula_version': CHOI_FORMULA_VERSION, 'observable': OBSERVABLE,
        'particle_transfer_definition': PARTICLE_TRANSFER_DEFINITION, 'gap_definition': GAP_DEFINITION,
        'gap_extraction': GAP_EXTRACTION, 'final_spectrum_extraction': FINAL_SPECTRUM_EXTRACTION,
        'batch_size': int(result['batch_size']), 'batch_size_mode': result['batch_size_mode'],
        'batch_size_auto_info': result.get('batch_size_auto_info'), 'retried_after_oom': bool(retried_after_oom),
        'lyapunov_modes_saved': 'all finite final-cycle transfer modes per sample',
        'lyapunov_mode_axis_convention': 'finite_lyapunov_modes[:, k] is mode k in the reduced Choi slab basis',
        'elapsed_s': elapsed_s, 'Nlayer_full': int(model.Nlayer), 'Nlayer_active': int(observer.final_spectrum.shape[1]),
        'active_top_layer_indices': None if observer.active_top_layer_indices is None else observer.active_top_layer_indices.tolist(),
        'stable_samples_final': int(observer.stable_at_cycle[:, observer._cycle_to_index[cfg.cycles]].sum()),
        'stable_fraction_final': float(observer.stable_at_cycle[:, observer._cycle_to_index[cfg.cycles]].mean()),
        'first_failure_cycle': observer.first_failure_cycle.tolist(),
        'first_failure_reason': observer.first_failure_reason.tolist(),
        'observer_failure_details': observer.failure_details,
        'choi_diagnostics': result.get('choi_diagnostics'), 'aggregate_summary': summary,
    }
    row = {
        'config_id': f'N{cfg.nx}x{cfg.ny}_{DW_LABEL}_corr_{cfg.correction_label}_alpha_{alpha_tag(cfg.alpha_1)}',
        'Nx': cfg.nx, 'Ny': cfg.ny, 'cycles': cfg.cycles, 'alpha_1': cfg.alpha_1, 'alpha_2': cfg.alpha_2,
        'dw_label': DW_LABEL, 'dw_truncation': DW_TRUNCATION,
        'perfect_correction': cfg.perfect_correction, 'correction_label': cfg.correction_label,
        'sample_count': cfg.samples,
        'stable_samples_final': run_summary['stable_samples_final'],
        'transfer_gap_mean': summary['transfer_gap_mean'], 'transfer_gap_sem': summary['transfer_gap_sem'],
        'elapsed_s': elapsed_s, 'batch_size': int(result['batch_size']), 'batch_size_mode': result['batch_size_mode'],
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT, 'run_dir': str(run_dir),
    }
    write_json_atomic(run_dir / 'run_summary.json', run_summary)
    write_json_atomic(run_dir / 'latest_run.json', {
        'status': 'completed', 'config_id': row['config_id'], 'helper_version': HELPER_VERSION, 'run_index_row': row})
    print(f"[complete] {cfg.correction_label} Ny={cfg.ny} alpha={cfg.alpha_1:g} gap_mean={summary['transfer_gap_mean']:.6g} "
          f"stable={row['stable_samples_final']}/{cfg.samples} elapsed={elapsed_s:.1f}s batch={row['batch_size']}")
    del model, observer
    torch.cuda.empty_cache()
    return row


In [ ]:
campaign_started = time.perf_counter()
rows = []
campaign_bar = tqdm(total=len(CONFIGS), desc='Alpha-sweep configurations', unit='config', leave=True, dynamic_ncols=True)
try:
    for raw_cfg in CONFIGS:
        cfg = run_config_from_dict(raw_cfg)
        label = f'{cfg.correction_label} Ny={cfg.ny} alpha={cfg.alpha_1:g}'
        campaign_bar.set_postfix_str(f'{label} elapsed=0s', refresh=True)
        with progress_heartbeat(campaign_bar, label):
            rows.append(run_one_config(raw_cfg))
        campaign_bar.update(1)
finally:
    campaign_bar.close()
rows = sorted(rows, key=lambda row: (str(row['correction_label']), int(row['Ny']), float(row['alpha_1'])))
run_index_path = CAMPAIGN_ROOT / 'run_index.csv'
write_run_index(run_index_path, rows)
manifest = {
    'status': 'completed', 'campaign_name': CAMPAIGN_NAME, 'campaign_id': CAMPAIGN_ID,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT, 'helper_version': HELPER_VERSION,
    'bundle_root': str(BUNDLE_ROOT), 'gpu_data_root': str(GPU_DATA_ROOT),
    'output_root': str(OUTPUT_ROOT), 'campaign_root': str(CAMPAIGN_ROOT), 'runs_root': str(RUNS_ROOT),
    'run_index': str(run_index_path), 'configs': CONFIGS, 'results': rows,
    'elapsed_s': float(time.perf_counter() - campaign_started), 'gpu_target': 'A100 40GB',
    'dtype': DTYPE, 'backend': BACKEND, 'skip_completed': bool(SKIP_COMPLETED), 'overwrite': bool(OVERWRITE),
}
campaign_manifest_path = CAMPAIGN_ROOT / 'campaign_manifest.json'
write_json_atomic(campaign_manifest_path, manifest)
write_json_atomic(OUTPUT_ROOT / 'latest_campaign.json', {
    'campaign_name': CAMPAIGN_NAME, 'campaign_id': CAMPAIGN_ID,
    'manifest': str(campaign_manifest_path.relative_to(GPU_DATA_ROOT)),
    'campaign_manifest_path': str(campaign_manifest_path),
    'campaign_manifest_path_relative': str(campaign_manifest_path.relative_to(GPU_DATA_ROOT)),
})
update_gpu_data_index(campaign_manifest_path)
run_index = pd.DataFrame(rows)
display(Markdown('### Saved run index'))
display(run_index)
print(f'[done] campaign manifest: {campaign_manifest_path}')
print(f'[done] run index: {run_index_path}')
print(json.dumps(manifest, indent=2))


In [ ]:
try:
    display(Javascript('google.colab.kernel.disconnect()'))
except Exception as exc:
    print(f'[disconnect] skipped: {exc}')
